# Case Study: Sentiment Analysis using BERT

**Objective:** Fine-tune a pre-trained **BERT** (`bert-base-uncased`) model to classify movie reviews as **Positive** or **Negative**.

**Dataset:** IMDB Movie Reviews (50,000 reviews, labelled 0 = Negative, 1 = Positive) loaded from Hugging Face `datasets`.

**Workflow:**
1. Install & import libraries
2. Load and explore the dataset
3. Tokenize text with the BERT tokenizer
4. Build PyTorch DataLoaders
5. Fine-tune `BertForSequenceClassification`
6. Evaluate (accuracy, precision, recall, F1, confusion matrix)
7. Predict sentiment of custom sentences

> Run on **Google Colab with GPU**: `Runtime → Change runtime type → T4 GPU`.

## 1. Install & Import Libraries

In [1]:
!pip install -q transformers datasets scikit-learn

In [2]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
!pip install -U datasets huggingface_hub

  Using cached huggingface_hub-2.0.0-py3-none-any.whl.metadata (16 kB)


## 2. Load the IMDB Dataset

The full dataset has 25k train + 25k test reviews. To keep training time reasonable on Colab, we use a shuffled subset
(increase `TRAIN_SIZE` / `TEST_SIZE` for higher accuracy).

In [4]:
TRAIN_SIZE = 5000
TEST_SIZE = 1000

imdb = load_dataset("imdb", revision="main")
train_data = imdb["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
test_data = imdb["test"].shuffle(seed=SEED).select(range(TEST_SIZE))

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)
print("Train shape:", train_df.shape, "| Test shape:", test_df.shape)
train_df.head()

HfUriError: Invalid HF URI 'hf://datasets/imdb@e6281661ce1c48d982bc483cf8a173c1bbeb5d31/.huggingface.yaml'. Repository id must be 'namespace/name', got 'imdb'.

## 3. Exploratory Data Analysis

In [ ]:
label_names = {0: "Negative", 1: "Positive"}
train_df["sentiment"] = train_df["label"].map(label_names)
train_df["word_count"] = train_df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(x="sentiment", data=train_df, ax=axes[0], palette="Set2")
axes[0].set_title("Class Distribution (Train)")
sns.histplot(train_df["word_count"], bins=50, ax=axes[1], color="steelblue")
axes[1].axvline(256, color="red", linestyle="--", label="max_len = 256")
axes[1].set_title("Review Length (words)")
axes[1].legend()
plt.tight_layout(); plt.show()

print(train_df["sentiment"].value_counts())
print("\nAverage words per review:", round(train_df["word_count"].mean(), 1))

In [ ]:
for label in [0, 1]:
    print(f"--- Sample {label_names[label]} review ---")
    print(train_df[train_df.label == label].iloc[0]["text"][:400], "...\n")

## 4. Tokenization with BERT Tokenizer

BERT uses WordPiece tokenization. Each review is converted to:
- `input_ids` – token ids with `[CLS]` at start and `[SEP]` at end
- `attention_mask` – 1 for real tokens, 0 for padding

In [ ]:
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

example = "This movie was absolutely fantastic!"
print("Tokens   :", tokenizer.tokenize(example))
print("Input IDs:", tokenizer.encode(example))

In [ ]:
def encode(texts):
    enc = tokenizer(list(texts), padding="max_length", truncation=True,
                    max_length=MAX_LEN, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]

train_ids, train_mask = encode(train_df["text"])
test_ids, test_mask = encode(test_df["text"])
train_labels = torch.tensor(train_df["label"].values)
test_labels = torch.tensor(test_df["label"].values)

print("Train input_ids shape:", train_ids.shape)
print("Test  input_ids shape:", test_ids.shape)

## 5. Create DataLoaders

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(TensorDataset(train_ids, train_mask, train_labels),
                          batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(test_ids, test_mask, test_labels),
                         batch_size=BATCH_SIZE)
print("Train batches:", len(train_loader), "| Test batches:", len(test_loader))

## 6. Load Pre-trained BERT Model

`BertForSequenceClassification` = BERT encoder + a linear classification layer on top of the `[CLS]` token output.

In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

EPOCHS = 2
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps),
                                            num_training_steps=total_steps)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. Fine-tune the Model

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, labels, total_loss = [], [], 0
    with torch.no_grad():
        for ids, mask, y in loader:
            ids, mask, y = ids.to(device), mask.to(device), y.to(device)
            out = model(input_ids=ids, attention_mask=mask, labels=y)
            total_loss += out.loss.item()
            preds.extend(out.logits.argmax(dim=1).cpu().numpy())
            labels.extend(y.cpu().numpy())
    return total_loss / len(loader), accuracy_score(labels, preds), np.array(preds), np.array(labels)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, seen = 0, 0, 0
    for ids, mask, y in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}"):
        ids, mask, y = ids.to(device), mask.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=y)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += out.loss.item()
        correct += (out.logits.argmax(dim=1) == y).sum().item()
        seen += y.size(0)

    train_loss, train_acc = total_loss / len(train_loader), correct / seen
    val_loss, val_acc, _, _ = evaluate(model, test_loader)
    for k, v in zip(history, [train_loss, train_acc, val_loss, val_acc]):
        history[k].append(v)
    print(f"Epoch {epoch + 1}: train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
          f"| val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history["train_loss"], "o-", label="Train")
axes[0].plot(epochs, history["val_loss"], "o-", label="Validation")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(epochs, history["train_acc"], "o-", label="Train")
axes[1].plot(epochs, history["val_acc"], "o-", label="Validation")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 8. Evaluation on Test Set

In [ ]:
test_loss, test_acc, y_pred, y_true = evaluate(model, test_loader)
print(f"Test Accuracy: {test_acc * 100:.2f}%\n")
print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"], digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()

## 9. Predict Sentiment of Custom Text

In [ ]:
def predict_sentiment(text):
    model.eval()
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=1)[0].cpu().numpy()
    label = int(probs.argmax())
    return label_names[label], float(probs[label])

samples = [
    "This movie was a masterpiece, the acting and story were brilliant!",
    "Worst film I have ever seen. Total waste of time and money.",
    "The plot was okay but the ending really disappointed me.",
    "I loved the music and the visuals, would definitely watch again.",
    "Boring, slow and predictable. I fell asleep halfway through.",
]
for s in samples:
    sentiment, conf = predict_sentiment(s)
    print(f"{sentiment:8s} ({conf * 100:5.1f}%)  ->  {s}")

## 10. Save the Fine-tuned Model

In [ ]:
SAVE_DIR = "bert_sentiment_model"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Model saved to", SAVE_DIR)

## 11. Conclusion

- We fine-tuned the pre-trained **BERT-base-uncased** transformer for binary sentiment classification on IMDB movie reviews.
- Using only a small subset (5,000 reviews) and 2 epochs, BERT reaches roughly **88–92% test accuracy**, because it already
  understands language context from pre-training (transfer learning).
- Unlike bag-of-words models, BERT is **bidirectional**: it reads the whole sentence at once, so it handles negation and
  context (e.g. *"not good"*, *"okay but disappointing"*) much better.
- Accuracy can be further improved by training on the full 25k dataset, more epochs, or a larger `MAX_LEN`.